In [ ]:
from datetime import datetime
import requests
import pandas as pd
import xml.etree.ElementTree as ET
import io
import boto3

from airflow.sdk import dag, task

VALUTES = ('USD', 'EUR', 'GBP', 'CNY', 'JPY', 'CHF', 'HKD', 'TRY')

@dag(
    schedule='@daily',
    start_time=datetime(2026, 1, 1),
    catchup=False
)

def cbr_currencies_pipeline(**context):

    @task
    def pull_from_api(date: str) -> list[dict]:   
        
        BASE_URL = 'http://www.cbr.ru/scripts/XML_daily.asp?date_req='
        URL = f'{BASE_URL}{date}'

        response = requests.get(URL)

        root = ET.fromstring(response.text)

        raw_data = []
        for valute in root:
            raw_data.append({
                'char_code': valute.find('CharCode').text,
                'name': valute.find('Name').text,
                'nominal': valute.find('Nominal').text,
                'value': valute.find('Value').text,
            })

        return raw_data

    @task
    def transform_raw_to_buffer(raw_data: list[dict], date: str) -> bytes:

        df_raw = pd.DataFrame(raw_data)

        df_filtered = df_raw[df_raw['char_code'].isin(VALUTES)].copy()
        df_filtered['nominal'] = df_filtered['nominal'].astype(int)
        df_filtered['value'] = df_filtered['value'].str.replace(',', '.').astype(float)

        result = df_filtered.assign(date = pd.to_datetime(date, format='%d/%m/%Y'))

        buffer = io.BytesIO()
        result.to_parquet(buffer)
        buffer.seek(0)

        return buffer

    @task
    def push_to_s3(buffer: bytes, date: str):

        s3_client = boto3.client(
            "s3",
            endpoint_url='http://localhost:9000',
            aws_access_key_id='minioadmin',
            aws_secret_access_key='minioadmin',
            region_name='eu-west-1'
        )

        try:
            s3_client.create_bucket(Bucket='raw')
        except s3_client.exceptions.BucketAlreadyOwnedByYou:
            pass

        s3_key = f'raw/currencies/date={pd.to_datetime(date, format="%d/%m/%Y").date()}/currencies.parquet'

        s3_client.upload_fileobj(
            Fileobj=buffer,
            Bucket='raw',
            Key=s3_key
        )

    date = context['ds']

    raw_data = pull_from_api(date)
    buffer_bytes = transform_raw_to_buffer(raw_data, date)
    push_to_s3(buffer_bytes, date)

cbr_currencies_pipeline()

In [ ]:
WITH all_ships AS (
    SELECT
        ship AS name,
        COALESCE(s.class, o.ship) AS class
    FROM Outcomes AS o
    LEFT JOIN Ships AS s
        ON o.ship = s.name

    UNION

    SELECT name, class
    FROM Ships
),
ships_ranked AS (
    SELECT *,    
        RANK() OVER (
            PARTITION BY displacement
            ORDER BY bore DESC
        ) AS rnk
    FROM all_ships AS a
    INNER JOIN Classes AS c
        ON a.class = c.class
)
SELECT name
FROM ships_ranked
WHERE rnk = 1
